In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [40]:
# load the dataset
path_to_datasets = Path().cwd().parent / "dataset"
train_df = pd.read_csv(path_to_datasets / "train.csv")
test_df = pd.read_csv(path_to_datasets / "test.csv")

In [41]:
# lets start by dropping the required columns
train_df.drop(columns=['id','recorded_by','num_private', 'scheme_name','wpt_name', 'subvillage',
               'ward', 'region_code', 'district_code', 'quantity_group',
               'payment_type', 'quality_group', 'source_type', 'waterpoint_type_group',
               'extraction_type_group', 'management_group', 'public_meeting', 'scheme_management', 'funder', 'installer'], 
               inplace=True)

# do the same for test set
test_df.drop(columns=['id','recorded_by','num_private', 'scheme_name','wpt_name', 'subvillage',
               'ward', 'region_code', 'district_code', 'quantity_group',
               'payment_type', 'quality_group', 'source_type', 'waterpoint_type_group',
               'extraction_type_group', 'management_group', 'public_meeting', 'scheme_management', 'funder', 'installer'], 
               inplace=True)

In [42]:
train_df.shape, test_df.shape

((59400, 21), (14850, 20))

In [43]:
print("Shape of the training dataset after dropping columns:", train_df.shape)
print("Shape of the test dataset after dropping columns:", test_df.shape)

Shape of the training dataset after dropping columns: (59400, 21)
Shape of the test dataset after dropping columns: (14850, 20)


In [44]:
print(train_df.dtypes)

amount_tsh               float64
date_recorded                str
gps_height                 int64
longitude                float64
latitude                 float64
basin                        str
region                       str
lga                          str
population                 int64
permit                    object
construction_year          int64
extraction_type              str
extraction_type_class        str
management                   str
payment                      str
water_quality                str
quantity                     str
source                       str
source_class                 str
waterpoint_type              str
status_group                 str
dtype: object


In [45]:
print(train_df.columns)

Index(['amount_tsh', 'date_recorded', 'gps_height', 'longitude', 'latitude',
       'basin', 'region', 'lga', 'population', 'permit', 'construction_year',
       'extraction_type', 'extraction_type_class', 'management', 'payment',
       'water_quality', 'quantity', 'source', 'source_class',
       'waterpoint_type', 'status_group'],
      dtype='str')


In [46]:
# number of categorical vs numerical features
print(f"Number of numerical features in train_df is {len(train_df.select_dtypes(include = ['number']).columns.to_list())}")
print(f"Number of categorical features in train_df is {len(train_df.select_dtypes(include = ['str', 'object']).columns.to_list())}")

# should be the same
print(f"Number of numerical features in test_df is {len(test_df.select_dtypes(include = ['number']).columns.to_list())}")
print(f"Number of categorical features in test_df is {len(test_df.select_dtypes(include = ['str', 'object']).columns.to_list())}")

Number of numerical features in train_df is 6
Number of categorical features in train_df is 15
Number of numerical features in test_df is 6
Number of categorical features in test_df is 14


### Checking missing values per feature:

In [47]:
train_df.isnull().sum()

amount_tsh                  0
date_recorded               0
gps_height                  0
longitude                   0
latitude                    0
basin                       0
region                      0
lga                         0
population                  0
permit                   3056
construction_year           0
extraction_type             0
extraction_type_class       0
management                  0
payment                     0
water_quality               0
quantity                    0
source                      0
source_class                0
waterpoint_type             0
status_group                0
dtype: int64

In [48]:
test_df.isnull().sum()

amount_tsh                 0
date_recorded              0
gps_height                 0
longitude                  0
latitude                   0
basin                      0
region                     0
lga                        0
population                 0
permit                   737
construction_year          0
extraction_type            0
extraction_type_class      0
management                 0
payment                    0
water_quality              0
quantity                   0
source                     0
source_class               0
waterpoint_type            0
dtype: int64

### Checking duplicate rows:

In [49]:
train_df.duplicated().sum()

np.int64(777)

In [50]:
# drop duplicates in train_df

# NOTE THAT WE HAVE 270 DUPLICATES IN TEST_DF WHICH MEANS THAT SINCE WE DROPPED OTHER FEATURES IN TEST
# THIS CAN ACCORDINGLY MAKE AN ERROR IN THESE 270 ROWS UNLESS THEY WERE TRULY DUPLICATE PRE-DROPPING 
train_df.drop_duplicates(inplace=True)
train_df.duplicated().sum()

np.int64(0)

### Cardinality of categorical features:

In [51]:
print(train_df.select_dtypes(include=['object', 'str']).nunique())

date_recorded            356
basin                      9
region                    21
lga                      125
permit                     2
extraction_type           18
extraction_type_class      7
management                12
payment                    7
water_quality              8
quantity                   5
source                    10
source_class               3
waterpoint_type            7
status_group               3
dtype: int64


In [52]:
# I WANT TO MAKE SURE THAT THERE ARE NO NEW WEIRD AHHH FEATURES IN THE TEST SET THAN THE ONES IN TRAINING SET
# firstly create a set out of the unique values in lga

print('diff in lga is: ',set(train_df['lga'].unique().tolist()) - set(test_df['lga'].unique().tolist()))
print('diff in region is: ',set(train_df['region'].unique().tolist()) - set(test_df['region'].unique().tolist()))

diff in lga is:  set()
diff in region is:  set()


### Statistical summary for numerical and categorical data:

In [53]:
train_df.describe(include='all')

,amount_tsh,date_recorded,gps_height,longitude,latitude,basin,region,lga,population,permit,...,extraction_type,extraction_type_class,management,payment,water_quality,quantity,source,source_class,waterpoint_type,status_group
count,58623.000000,58623,58623.000000,58623.000000,5.862300e+04,58623,58623,58623,58623.000000,55567,...,58623,58623,58623,58623,58623,58623,58623,58623,58623,58623
unique,NaN,356,NaN,NaN,NaN,9,21,125,NaN,2,...,18,7,12,7,8,5,10,3,7,3
top,NaN,2011-03-15,NaN,NaN,NaN,Lake Victoria,Iringa,Njombe,NaN,True,...,gravity,gravity,vwc,never pay,soft,enough,spring,groundwater,communal standpipe,functional
freq,NaN,572,NaN,NaN,NaN,9513,5294,2503,NaN,38525,...,26741,26741,40179,24944,50180,32748,17015,45117,28466,31818
mean,321.860581,NaN,677.154973,34.528556,-5.781631e+00,NaN,NaN,NaN,182.294543,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,3017.150144,NaN,693.382772,5.303495,2.890864e+00,NaN,NaN,NaN,474.138310,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,NaN,-90.000000,0.000000,-1.164944e+01,NaN,NaN,NaN,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,NaN,0.000000,33.167410,-8.579859e+00,NaN,NaN,NaN,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,NaN,387.000000,34.946453,-5.083189e+00,NaN,NaN,NaN,30.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,25.000000,NaN,1325.000000,37.201057,-3.344150e+00,NaN,NaN,NaN,225.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Start: Replacing 0 With NaN (Following merged_conc)

In [54]:
train_df.isnull().sum()

amount_tsh                  0
date_recorded               0
gps_height                  0
longitude                   0
latitude                    0
basin                       0
region                      0
lga                         0
population                  0
permit                   3056
construction_year           0
extraction_type             0
extraction_type_class       0
management                  0
payment                     0
water_quality               0
quantity                    0
source                      0
source_class                0
waterpoint_type             0
status_group                0
dtype: int64

In [55]:
test_df.isnull().sum()

amount_tsh                 0
date_recorded              0
gps_height                 0
longitude                  0
latitude                   0
basin                      0
region                     0
lga                        0
population                 0
permit                   737
construction_year          0
extraction_type            0
extraction_type_class      0
management                 0
payment                    0
water_quality              0
quantity                   0
source                     0
source_class               0
waterpoint_type            0
dtype: int64

In [56]:
# NOTICE THAT THERE ARE SOME CONSTRUCTION YEARS THAT ARE NULLS 

### Missing Values Imputation

In [57]:
# amount_tsh == amount total static head
train_df.groupby('status_group')['amount_tsh'].apply(lambda x : x.isna().mean())

status_group
functional                 0.0
functional needs repair    0.0
non functional             0.0
Name: amount_tsh, dtype: float64

In [58]:
# lets look at the unique values in permit column

train_df['permit'].unique()

# probably not having permission would make the water pump more likely to be non functional
# lets look at the percentage of missing values in permit column for each status group
train_df.groupby('status_group')['permit'].apply(lambda x : x.isna().mean())

status_group
functional                 0.052580
functional needs repair    0.071582
non functional             0.047891
Name: permit, dtype: float64

In [59]:
# well there is no difference so that was not the case
# however ill not fill with mode as this is not a good idea to fill and assume that it had permission to be made
# without proper evidence
# so instead ill fill with unknown
# as much as this would increase the cardinality
# as much as it will still not be a bad idea since we will be using tree models

train_df['permit'] = train_df['permit'].astype(str)
test_df['permit'] = test_df['permit'].astype(str)

In [60]:
print(train_df.isnull().sum())

amount_tsh                  0
date_recorded               0
gps_height                  0
longitude                   0
latitude                    0
basin                       0
region                      0
lga                         0
population                  0
permit                   3056
construction_year           0
extraction_type             0
extraction_type_class       0
management                  0
payment                     0
water_quality               0
quantity                    0
source                      0
source_class                0
waterpoint_type             0
status_group                0
dtype: int64


In [61]:
print(test_df.isnull().sum())

amount_tsh                 0
date_recorded              0
gps_height                 0
longitude                  0
latitude                   0
basin                      0
region                     0
lga                        0
population                 0
permit                   737
construction_year          0
extraction_type            0
extraction_type_class      0
management                 0
payment                    0
water_quality              0
quantity                   0
source                     0
source_class               0
waterpoint_type            0
dtype: int64


In [62]:
# now lets one hot encode the rest of the features
# lets first recheck the nunique of each
train_df.select_dtypes(include = ['str', 'object']).nunique()

date_recorded            356
basin                      9
region                    21
lga                      125
permit                     2
extraction_type           18
extraction_type_class      7
management                12
payment                    7
water_quality              8
quantity                   5
source                    10
source_class               3
waterpoint_type            7
status_group               3
dtype: int64

In [63]:
# NOW CAPTURE THE MONTH AND YEAR RECORDED
train_df['year_recorded'] = pd.to_datetime(train_df['date_recorded']).dt.year
train_df['month_recorded'] = pd.to_datetime(train_df['date_recorded']).dt.month


test_df['year_recorded'] = pd.to_datetime(test_df['date_recorded']).dt.year
test_df['month_recorded'] = pd.to_datetime(test_df['date_recorded']).dt.month

train_df.drop(columns = ['date_recorded'], inplace = True)
test_df.drop(columns = ['date_recorded'], inplace = True)

In [64]:
from sklearn.preprocessing import OneHotEncoder

cat_features = [value for value in train_df.select_dtypes(include = ['object', 'str']).columns if value not in ['status_group', 'source']]

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

train_df_encoded = ohe.fit_transform(train_df[cat_features]).astype(int)
test_df_encoded = ohe.transform(test_df[cat_features]).astype(int)  #type: ignore

# convert back to being a dataframe and concatenate with original df
train_df_encoded = pd.DataFrame(train_df_encoded, columns=ohe.get_feature_names_out(cat_features))
test_df_encoded = pd.DataFrame(test_df_encoded, columns=ohe.get_feature_names_out(cat_features))
train_df = pd.concat([train_df.reset_index(drop=True), train_df_encoded.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), test_df_encoded.reset_index(drop=True)], axis=1)

train_df.drop(columns=cat_features, inplace=True)
test_df.drop(columns=cat_features, inplace=True)

In [65]:
train_df.shape, test_df.shape

((58623, 235), (14850, 234))

In [66]:

num_cols = ['amount_tsh', 'gps_height', 'longitude', 'latitude',  # original
            'population', 'construction_year', 'year_recorded', 'month_recorded']

print("Outlier check after scaling (IQR method):")
for col in num_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((train_df[col] < lower) | (train_df[col] > upper)).sum()
    print(f"  {col}: {n_out} outliers ({n_out/len(train_df)*100:.1f}%)")

Outlier check after scaling (IQR method):
  amount_tsh: 11144 outliers (19.0%)
  gps_height: 0 outliers (0.0%)
  longitude: 1036 outliers (1.8%)
  latitude: 0 outliers (0.0%)
  population: 4024 outliers (6.9%)
  construction_year: 0 outliers (0.0%)
  year_recorded: 31 outliers (0.1%)
  month_recorded: 0 outliers (0.0%)


In [67]:
# lets create a pump_age feature by subtracting the construction_year from the year_recorded
# we need to use the original year_recorded before scaling for this calculation
# we need to revert the robust scaling for year_recorded and construction_year to do this calculation correctly
# to do so 
# the robust scaling = (x - median) / IQR
# so to revert it we do: x = (scaled_x * IQR) + median
# however we dont have the iqr nor the mean from the original feature so ill just comment the part above for now

train_df['pump_age'] = train_df['year_recorded'] - train_df['construction_year']
test_df['pump_age'] = test_df['year_recorded'] - test_df['construction_year']

train_df['pump_age']

0          12
1           3
2           4
3          27
4        2011
         ... 
58618      14
58619      15
58620    2011
58621    2011
58622       9
Name: pump_age, Length: 58623, dtype: int64

In [68]:
train_df['pump_age'].skew()

np.float64(0.6751603186384105)

In [69]:
train_df['pump_age'].kurtosis()

np.float64(-1.5437018876920894)

In [70]:
train_df.drop(columns = ['source'], inplace = True)
test_df.drop(columns = ['source'], inplace = True)

In [71]:
train_df.head()

,amount_tsh,gps_height,longitude,latitude,population,construction_year,status_group,year_recorded,month_recorded,basin_Internal,...,source_class_surface,source_class_unknown,waterpoint_type_cattle trough,waterpoint_type_communal standpipe,waterpoint_type_communal standpipe multiple,waterpoint_type_dam,waterpoint_type_hand pump,waterpoint_type_improved spring,waterpoint_type_other,pump_age
0,6000.0,1390,34.938093,-9.856322,109,1999,functional,2011,3,0,...,0,0,0,1,0,0,0,0,0,12
1,0.0,1399,34.698766,-2.147466,280,2010,functional,2013,3,0,...,1,0,0,1,0,0,0,0,0,3
2,25.0,686,37.460664,-3.821329,250,2009,functional,2013,2,0,...,1,0,0,0,1,0,0,0,0,4
3,0.0,263,38.486161,-11.155298,58,1986,non functional,2013,1,0,...,0,0,0,0,1,0,0,0,0,27
4,0.0,0,31.130847,-1.825359,0,0,functional,2011,7,0,...,1,0,0,1,0,0,0,0,0,2011


In [72]:
test_df.head()

,amount_tsh,gps_height,longitude,latitude,population,construction_year,year_recorded,month_recorded,basin_Internal,basin_Lake Nyasa,...,source_class_surface,source_class_unknown,waterpoint_type_cattle trough,waterpoint_type_communal standpipe,waterpoint_type_communal standpipe multiple,waterpoint_type_dam,waterpoint_type_hand pump,waterpoint_type_improved spring,waterpoint_type_other,pump_age
0,0.0,1996,35.290799,-4.059696,321,2012,2013,2,1,0,...,1,0,0,0,0,0,0,0,1,1
1,0.0,1569,36.656709,-3.309214,300,2000,2013,2,0,0,...,0,0,0,1,0,0,0,0,0,13
2,0.0,1567,34.767863,-5.004344,500,2010,2013,2,1,0,...,1,0,0,0,0,0,0,0,1,3
3,0.0,267,38.058046,-9.418672,250,1987,2013,1,0,0,...,0,0,0,0,0,0,0,0,1,26
4,500.0,1260,35.006123,-10.950412,60,2000,2013,3,0,0,...,0,0,0,1,0,0,0,0,0,13


In [73]:
test_df.isnull().sum()

amount_tsh                         0
gps_height                         0
longitude                          0
latitude                           0
population                         0
                                  ..
waterpoint_type_dam                0
waterpoint_type_hand pump          0
waterpoint_type_improved spring    0
waterpoint_type_other              0
pump_age                           0
Length: 234, dtype: int64

In [74]:
# save the preprocessed datasets
train_df.to_csv(path_to_datasets / "train_clean.csv", index=False)
test_df.to_csv(path_to_datasets / "test_clean.csv", index=False)